In [12]:
import pandas as pd
import numpy as np
import joblib
import random

import matplotlib.pyplot as plt


In [13]:
aliens = pd.read_csv("../data/processed/aliens_with_archetypes.csv")
battle_model = joblib.load("../model/battle_model.pkl")
alien_stats = pd.read_csv("../data/ben10_aliens.csv")


In [14]:
def build_battle_features(alien_a, alien_b, stats_df):
    A = stats_df[stats_df["alien_name"] == alien_a].iloc[0]
    B = stats_df[stats_df["alien_name"] == alien_b].iloc[0]

    strength_diff = A["strength_level"] - B["strength_level"]
    speed_diff = A["speed_level"] - B["speed_level"]
    intelligence_diff = A["intelligence"] - B["intelligence"]

    features = {
        "strength_diff": strength_diff,
        "speed_diff": speed_diff,
        "intelligence_diff": intelligence_diff,
        "strength_gap": abs(strength_diff),
        "speed_gap": abs(speed_diff),
        "intelligence_gap": abs(intelligence_diff),

        "strength_ratio": A["strength_level"] / (B["strength_level"] + 1),
        "speed_ratio": A["speed_level"] / (B["speed_level"] + 1),
        "intelligence_ratio": A["intelligence"] / (B["intelligence"] + 1),

        "power_diff": (
            (A["strength_level"] + A["speed_level"]) -
            (B["strength_level"] + B["speed_level"])
        ),

        "total_diff": (
            (A["strength_level"] + A["speed_level"] + A["intelligence"]) -
            (B["strength_level"] + B["speed_level"] + B["intelligence"])
        ),

        "max_stat_diff": max(
            A["strength_level"], A["speed_level"], A["intelligence"]
        ) - max(
            B["strength_level"], B["speed_level"], B["intelligence"]
        ),

        "min_stat_diff": min(
            A["strength_level"], A["speed_level"], A["intelligence"]
        ) - min(
            B["strength_level"], B["speed_level"], B["intelligence"]
        ),

        "variance_diff": np.var(
            [A["strength_level"], A["speed_level"], A["intelligence"]]
        ) - np.var(
            [B["strength_level"], B["speed_level"], B["intelligence"]]
        ),

        "speed_advantage": int(speed_diff > 0),
        "speed_dominance": speed_diff * (A["speed_level"] / (B["speed_level"] + 1))
    }

    return pd.DataFrame([features])


In [15]:
def simulate_battle(alien_a, alien_b):
    X = build_battle_features(alien_a, alien_b, alien_stats)
    win_prob = battle_model.predict_proba(X)[0][1]

    winner = alien_a if random.random() < win_prob else alien_b
    return winner, win_prob


In [16]:
def run_tournament(alien_list):
    current_round = alien_list.copy()
    random.shuffle(current_round)

    while len(current_round) > 1:
        next_round = []

        for i in range(0, len(current_round), 2):
            if i + 1 >= len(current_round):
                next_round.append(current_round[i])
                continue

            a, b = current_round[i], current_round[i + 1]
            winner, _ = simulate_battle(a, b)
            next_round.append(winner)

        current_round = next_round

    return current_round[0]


In [ ]:
N_TOURNAMENTS = 500

winners = []

alien_names = aliens["alien_name"].tolist()

for _ in range(N_TOURNAMENTS):
    champion = run_tournament(alien_names)
    winners.append(champion)


In [ ]:
winner_counts = pd.Series(winners).value_counts()


In [ ]:
#who statistically dominated the the simulation
winner_counts.head(10)

In [ ]:
winner_df = winner_counts.reset_index()
winner_df.columns = ["alien_name", "wins"]

winner_df = winner_df.merge(
    aliens[["alien_name", "archetype"]],
    on="alien_name"
)


In [ ]:
archetype_wins = winner_df.groupby("archetype")["wins"].sum()

plt.figure()
archetype_wins.sort_values().plot(kind="barh")
plt.title("Tournament Wins by Archetype")
plt.xlabel("Number of Championships")
plt.show()


In [ ]:
cumulative = pd.Series(dtype=int)

for i in range(1, N_TOURNAMENTS + 1):
    champ = run_tournament(alien_names)
    cumulative[champ] = cumulative.get(champ, 0) + 1

    if i % 100 == 0:
        print(f"After {i} tournaments:")
        print(cumulative.sort_values(ascending=False).head(5))
